# MCTS Eval (Colab) — pillar2y2 + q_weight=0.5 sweep

Compare pillar2y2 (V11) vs pillar2x2 (V10) at 800 sims with the new
**q_weight=0.5** PUCT calibration. q_weight=1.0 (the old default) gave
Q the same dynamic range as a perfect signal even though feature-value
regression has r≈0.5 with truth — overweighting it hurt MCTS on
distilled-policy generations. q=0.5 was the sweet spot in the local
50-seed sweep (mean +20% / median +54% over q=1.0 on 2Y2).

**Critical:** BATCH_SIZE=8 (HISTORY lesson 115 — virtual-loss starvation
kills MCTS quality at bs > 8).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (must include the q_weight commit)
2. `feature_value_weights_2y.npz` — V11-fit feature evaluator
3. `feature_value_weights_2x.npz` — V10-fit (for the 2X2 baseline)
4. The checkpoints to compare


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy both weight files (each checkpoint pairs with its own V-iteration)
os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/feature_value_weights_2y.npz /content/alphatrain/data/feature_value_weights_2y.npz
!cp {DRIVE}/feature_value_weights_2x.npz /content/alphatrain/data/feature_value_weights_2x.npz
for f in ['feature_value_weights_2y.npz', 'feature_value_weights_2x.npz']:
    print(f'{f}: {os.path.getsize(f"/content/alphatrain/data/{f}")} bytes')

!pip install -q numpy numba scipy

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# === EDIT THESE ===
# Each (label, model, weights) triple is one full eval run.
# Default: 2Y2 (V11, q=0.5 sweet spot) vs 2X2 baseline (V10).
RUNS = [
    ('2Y2 ep40 + _2y',  f'{DRIVE}/pillar2y2_epoch_40.pt',
                        '/content/alphatrain/data/feature_value_weights_2y.npz'),
    ('2X2 ep10 + _2x',  f'{DRIVE}/pillar2x2_epoch_10.pt',
                        '/content/alphatrain/data/feature_value_weights_2x.npz'),
]
SIMS = 800
Q_WEIGHT = 0.5         # PUCT score = q_weight * q_norm + U.
SEEDS = ' '.join(str(i) for i in range(50))   # 0..49
WORKERS = 24
BATCH_SIZE = 8         # MCTS quality requires bs=8 (HISTORY lesson 115)
MAX_TURNS = 5000       # cap (= ~10,500 score ceiling at this strength)
# ==================

for label, path, weights in RUNS:
    assert os.path.exists(path), f'MISSING model: {path}'
    assert os.path.exists(weights), f'MISSING weights: {weights}'
    print(f'OK: {label}  model={os.path.getsize(path)/1e6:.0f} MB')


In [ ]:
%cd /content

for label, path, weights in RUNS:
    print(f'\n{"="*60}')
    print(f'=== {label}: {path.split("/")[-1]} ===')
    print(f'{"="*60}\n', flush=True)
    !python -m alphatrain.scripts.eval_parallel \
        --model {path} \
        --feature-value-weights {weights} \
        --device cuda --workers {WORKERS} --batch-size {BATCH_SIZE} \
        --simulations {SIMS} --q-weight {Q_WEIGHT} \
        --max-turns {MAX_TURNS} \
        --early-stop --compile \
        --seeds {SEEDS}
